# Analisis Data Eksploratif (EDA) - Emotion Detection (Bahasa Indonesia)

Notebook ini bertujuan untuk mengeksplorasi dataset **IndoNLU EmoT** yang berisi tweet dalam Bahasa Indonesia yang dilabeli dengan 5 emosi berbeda: *sadness, anger, love, fear, happy*. Kita akan menganalisis:
1. Jumlah sampel data pada tiap split (train, validation, test).
2. Distribusi label emosi pada dataset.
3. Contoh teks tweet untuk masing-masing kelas emosi.
4. Distribusi panjang teks (jumlah kata) per tweet.

In [ ]:
import sys
import os

# Menambahkan path root project agar modul src bisa di-import
sys.path.append(os.path.abspath(".."))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data import load_emot_dataset, get_label_distribution

# Mengatur style visualisasi
sns.set_theme(style="whitegrid")

## 1. Load Dataset dan Ukuran Split

Kita memuat dataset `indonlp/indonlu` dengan konfigurasi `emot` dari Hugging Face Datasets.

In [ ]:
dataset = load_emot_dataset()
print("=== Dataset Split Info ===")
print(f"Train samples      : {len(dataset['train'])}")
print(f"Validation samples : {len(dataset['validation'])}")
print(f"Test samples       : {len(dataset['test'])}")

## 2. Distribusi Label Emosi

Mari kita lihat frekuensi tiap label di dataset training untuk mendeteksi potensi adanya ketidakseimbangan kelas (*class imbalance*).

In [ ]:
# Mendapatkan nama emosi
label_names = dataset["train"].features["label"].names
print(f"Daftar label emosi ({len(label_names)} kelas): {label_names}")

# Hitung distribusi label menggunakan helper get_label_distribution
train_dist = get_label_distribution(dataset["train"])

# Membuat DataFrame untuk visualisasi
df_dist = pd.DataFrame({
    "Label ID": list(train_dist.keys()),
    "Count": list(train_dist.values())
})
df_dist["Emotion"] = df_dist["Label ID"].apply(lambda x: label_names[x])
df_dist = df_dist.sort_values(by="Count", ascending=False)
print(df_dist)

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.barplot(data=df_dist, x="Emotion", y="Count", palette="viridis")
plt.title("Distribusi Kelas Emosi pada Dataset Training", fontsize=14)
plt.xlabel("Emosi", fontsize=12)
plt.ylabel("Jumlah Sampel", fontsize=12)

# Menambahkan label angka di atas bar
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', 
                xytext=(0, 5), 
                textcoords='offset points', 
                fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Contoh Tweet Per Kelas Emosi

Mari kita tampilkan beberapa contoh tweet asli untuk masing-masing kelas emosi.

In [ ]:
df_train = pd.DataFrame(dataset["train"])
df_train["Emotion"] = df_train["label"].apply(lambda x: label_names[x])

for label in label_names:
    print(f"\n=== EMOSI: {label.upper()} ===")
    samples = df_train[df_train["Emotion"] == label]["tweet"].head(5).tolist()
    for i, tweet in enumerate(samples, 1):
        print(f"{i}. {tweet}")
    print("="*60)

## 4. Statistik Panjang Teks (Jumlah Kata)

Kita hitung panjang teks (sederhana dengan pemisahan spasi) untuk melihat karakteristik panjang tweet.

In [ ]:
df_train["word_count"] = df_train["tweet"].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 5))
sns.histplot(data=df_train, x="word_count", kde=True, bins=30, color="teal")
plt.title("Distribusi Panjang Kata per Tweet (Training Set)", fontsize=14)
plt.xlabel("Jumlah Kata", fontsize=12)
plt.ylabel("Frekuensi", fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

print("=== Statistik Deskriptif Panjang Kata ===")
print(df_train["word_count"].describe())

## 5. Kesimpulan Analisis

Berdasarkan analisis di atas, diperoleh beberapa kesimpulan penting:
1. **Imbalance Dataset**:
   - Kelas emosi *anger* (1101 sampel), *love* (1043 sampel), dan *sadness* (1017 sampel) mendominasi dataset training.
   - Kelas emosi *fear* (622 sampel) dan *happy* (617 sampel) memiliki jumlah data yang lebih sedikit.
   - Terdapat **sedikit ketidakseimbangan kelas** (*moderate imbalance*), namun tidak sampai pada level ekstrem (rasio sekitar 1.8 : 1 antara kelas mayoritas vs minoritas). Oleh karena itu, *cross-entropy* standar tanpa *class weighting* umumnya sudah cukup baik dan stabil, namun metrik **Macro F1** tetap krusial sebagai acuan utama evaluasi karena memberikan bobot rata-rata yang setara pada tiap kelas tanpa memedulikan ukuran kelas.
2. **Karakteristik Teks**:
   - Tweet memiliki variasi panjang kata berkisar antara 2 kata hingga maksimal 62 kata, dengan rata-rata panjang teks sebesar 17 kata.
   - Panjang sequence `max_length = 96` yang kita rencanakan di pemrosesan data sudah sangat aman untuk meng-cover seluruh tweet tanpa ada informasi penting yang terpotong.